# Clase 03 - Del texto a representaciones numéricas

BoW, n-grams y TF-IDF. Sesión de 120 min; práctica 45 min. Antecedentes: Clases 1–2. No requiere terminar Actividad 1.

## Preparación
En Colab abre este notebook y sube el CSV indicado al panel Archivos.

In [1]:
from pathlib import Path
import sys, subprocess, importlib.util, json, platform, hashlib, re, unicodedata
from importlib.metadata import version
necesarios = {"numpy":"numpy==2.2.6", "pandas":"pandas==2.2.3",
              "sklearn":"scikit-learn==1.6.1", "skops":"skops==0.11.0",
              "matplotlib":"matplotlib==3.10.3"}
faltan = [paquete for modulo,paquete in necesarios.items()
          if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.run([sys.executable,"-m","pip","install",*faltan],check=True)
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import skops.io as sio
import matplotlib.pyplot as plt
versiones = {p:version(p) for p in necesarios.values() for p in [p.split("==")[0]]}
versiones["python"] = platform.python_version()
print(versiones)
completados = set()

{'numpy': '2.5.3', 'pandas': '3.0.5', 'scikit-learn': '1.9.0', 'skops': '0.11.0', 'matplotlib': '3.11.1', 'python': '3.12.10'}


## Datos y contrato
Lee `data/README.md`. Datos sintéticos originales, autorizados para docencia. `id` identifica el registro; `family` reúne variantes; `split` fija train/validation/test; `label` define acceso, pagos, conectividad o software; `text_original` conserva la fuente y `text_processed` aplica NFC y espacios.

Solo `text_processed` entra al modelo. Los identificadores y etiquetas no se añaden a las características. La clasificación orienta una revisión de soporte; no ejecuta acciones sobre cuentas ni pagos. Los ejemplos pequeños y repetitivos no representan toda la variedad del lenguaje real.


Archivo: corpus_clases_03_04.csv.

In [2]:
nombre = "corpus_clases_03_04.csv"
bases = [Path.cwd(), *Path.cwd().parents]
candidatos = [Path.cwd()/nombre]
for base in bases:
    candidatos += [base/"data"/nombre, base/"data"/nombre,
                   base/"data"/nombre]
ARCHIVO = next((p for p in candidatos if p.is_file()),None)
if ARCHIVO is None:
    raise FileNotFoundError("Extrae data/ junto al notebook o termina de subir el CSV al panel Archivos de Colab.")
df = pd.read_csv(ARCHIVO,keep_default_na=False,encoding="utf-8")
OUT = Path.cwd()/"resultados_clase03"
OUT.mkdir(exist_ok=True)
print("Fuente localizada:",ARCHIVO.name)

Fuente localizada: corpus_clases_03_04.csv


## 1. Inspeccionar
El control verifica columnas, IDs, ausencias, familias, textos y cobertura. Agrupar por family y contar nombres de split revela cruces. Leer etiquetas para comprobar cobertura no equivale a entrenar con ellas. No exploramos el rendimiento de test.

In [3]:
def comprobar_particiones(tabla):
    obligatorias = {"id","family","split","label","text_original","text_processed"}
    assert obligatorias <= set(tabla.columns), "Faltan columnas del contrato"
    assert tabla.id.is_unique, "IDs repetidos"
    assert not tabla[list(obligatorias)].isna().any().any(), "Hay ausencias"
    for col in obligatorias:
        assert tabla[col].astype(str).str.strip().ne("").all(), "Campo vacío: "+col
    assert set(tabla.split)=={"train","validation","test"}, "Revisar particiones"
    for col in ["family","text_processed"]:
        if tabla.groupby(col).split.nunique().max()>1:
            raise ValueError("Fuga por "+col)
    grupos = {s:tabla[tabla.split==s].copy() for s in ["train","validation","test"]}
    etiquetas = sorted(grupos["train"].label.unique())
    assert len(etiquetas)>=2
    for s,parte in grupos.items():
        assert set(parte.label)==set(etiquetas), "Cobertura distinta: "+s
    return grupos,etiquetas
partes, LABELS = comprobar_particiones(df)
train, val, test = [partes[s] for s in ["train","validation","test"]]
print(df.groupby("split").agg(filas=("id","size"),familias=("family","nunique")))
print(pd.crosstab(df.split,df.label).reindex(columns=LABELS))
completados.add(1)

            filas  familias
split                      
test           24         8
train          96        32
validation     24         8
label       acceso  conectividad  pagos  software
split                                            
test             6             6      6         6
train           24            24     24        24
validation       6             6      6         6


## 2. Cuenta
CountVectorizer cuenta términos; sus columnas están en get_feature_names_out. Cada fila corresponde a un documento. fit_transform aprende vocabulario y transforma esos mismos textos. El tokenizador por defecto conserva palabras de dos o más caracteres; lower convierte a minúsculas, pero strip_accents=None conserva diacríticos. No utiliza automáticamente las reglas de spaCy.

ngram_range=(1,2) añade pares contiguos. TF es el conteo crudo en este ejemplo; DF cuenta documentos con el término. La fórmula con suavizado es IDF=ln((1+n)/(1+DF))+1. norm=None permite inspeccionar TF×IDF; norm="l2" divide cada fila no nula por su longitud euclidiana. Esto no es normalización Unicode.

In [4]:
# Corpus diminuto para verificar cuentas, separado del experimento principal.
textos = ["pago pago cuenta", "pago tarjeta", "cuenta bloqueada"]
bow = CountVectorizer(lowercase=True,strip_accents=None,stop_words=None)
B = bow.fit_transform(textos)
nombres = bow.get_feature_names_out()
esperado = np.array([[0,1,2,0],[0,0,1,1],[1,1,0,0]])
assert list(nombres)==["bloqueada","cuenta","pago","tarjeta"]
assert np.array_equal(B.toarray(),esperado)
print(pd.DataFrame(B.toarray(),columns=nombres))
print("Tokens:",bow.build_analyzer()(textos[0]))
bi = CountVectorizer(ngram_range=(1,2),strip_accents=None,stop_words=None)
G = bi.fit_transform(textos)
print("Unigramas:",B.shape,"no ceros:",B.nnz)
print("Unigramas + bigramas:",G.shape,"no ceros:",G.nnz)
print(bi.get_feature_names_out())
print("Negación:",bi.build_analyzer()("no quiero cancelar"))
tf = TfidfVectorizer(norm=None,smooth_idf=True,sublinear_tf=False)
T = tf.fit_transform(textos)
frecuencia_documental = (B.toarray()>0).sum(axis=0)
idf_manual = np.log((1+len(textos))/(1+frecuencia_documental))+1
assert np.allclose(tf.idf_,idf_manual)
assert np.allclose(T.toarray(),B.toarray()*idf_manual)
tf_l2 = TfidfVectorizer(norm="l2",smooth_idf=True,sublinear_tf=False)
N = tf_l2.fit_transform(textos)
manual_l2 = T.toarray()/np.linalg.norm(T.toarray(),axis=1,keepdims=True)
assert np.allclose(N.toarray(),manual_l2)
print(pd.DataFrame({"termino":nombres,"DF":frecuencia_documental,"IDF":idf_manual}))
print("TF-IDF sin L2:",T.toarray().round(6))
print("TF-IDF con L2:",N.toarray().round(6))

   bloqueada  cuenta  pago  tarjeta
0          0       1     2        0
1          0       0     1        1
2          1       1     0        0
Tokens: ['pago', 'pago', 'cuenta']
Unigramas: (3, 4) no ceros: 6
Unigramas + bigramas: (3, 8) no ceros: 10
['bloqueada' 'cuenta' 'cuenta bloqueada' 'pago' 'pago cuenta' 'pago pago'
 'pago tarjeta' 'tarjeta']
Negación: ['no', 'quiero', 'cancelar', 'no quiero', 'quiero cancelar']
     termino  DF       IDF
0  bloqueada   1  1.693147
1     cuenta   2  1.287682
2       pago   2  1.287682
3    tarjeta   1  1.693147
TF-IDF sin L2: [[0.       1.287682 2.575364 0.      ]
 [0.       0.       1.287682 1.693147]
 [1.693147 1.287682 0.       0.      ]]
TF-IDF con L2: [[0.       0.447214 0.894427 0.      ]
 [0.       0.       0.605349 0.795961]
 [0.795961 0.605349 0.       0.      ]]


### Interpretación
La fila de pago pago cuenta es [0,1,2,0] con el vocabulario mostrado. pago aparece dos veces en esa fila, pero su DF es 2 porque está en dos documentos. El vector L2 [0,0.447214,0.894427,0] conserva la proporción 1:2 entre los términos presentes. Un peso alto es relativo a esta colección y representación; no prueba importancia causal.

## 3. Aplicar al corpus sin fuga
fit_transform recibe solo train. transform aplica columnas e IDF ya aprendidos a validación. shape devuelve filas y columnas; nnz cuenta valores distintos de cero. getrow toma una sola fila: inspeccionamos sus índices y pesos sin convertir toda la matriz en densa. OOV significa out of vocabulary: fuera del vocabulario aprendido.

In [5]:
vectorizador = TfidfVectorizer(ngram_range=(1,1),lowercase=True,
                              strip_accents=None,stop_words=None)
X_train = vectorizador.fit_transform(train.text_processed)
idf_antes = vectorizador.idf_.copy()
vocab_antes = vectorizador.vocabulary_.copy()
X_val = vectorizador.transform(val.text_processed)
assert vectorizador.vocabulary_==vocab_antes
assert np.array_equal(vectorizador.idf_,idf_antes)
analizar = vectorizador.build_analyzer()
solo_val = set(t for texto in val.text_processed for t in analizar(texto))-set(vocab_antes)
assert solo_val and solo_val.isdisjoint(vocab_antes)
print("Formas:",X_train.shape,X_val.shape,"no ceros train:",X_train.nnz)
print("Términos exclusivos de validación fuera del vocabulario:",sorted(solo_val)[:8])
fila = X_val.getrow(0)
tabla_fila = pd.DataFrame({"termino":vectorizador.get_feature_names_out()[fila.indices],
                         "peso":fila.data}).sort_values("peso",ascending=False)
print("ID interpretado:",val.iloc[0].id,"|",val.iloc[0].text_processed)
print(tabla_fila.head(8).to_string(index=False))
oov = vectorizador.transform(["hiperfoliozeta contraseña","hiperfoliozeta"])
assert oov.getrow(0).nnz>0 and oov.getrow(1).nnz==0
assert "hiperfoliozeta" not in vectorizador.vocabulary_
print("No ceros OOV parcial / total:",[oov.getrow(i).nnz for i in range(2)])

Formas: (96, 142) (24, 142) no ceros train: 1017
Términos exclusivos de validación fuera del vocabulario: ['anterior', 'aunque', 'autenticador', 'avisar', 'banco', 'botón', 'como', 'confirmó']
ID interpretado: S025 | La página regresa al inicio de sesión sin avisar.
termino     peso
 sesión 0.701346
     al 0.547914
     de 0.358462
     la 0.281798
No ceros OOV parcial / total: [1, 0]


## Reto propio - 15 minutos de los 45 de práctica
1. Elige otra fila del corpus de validación, identifica tres términos con peso distinto de cero e interpreta sus pesos. No atribuyas causalidad ni significado a un peso aislado.
2. Inventa dos textos con las mismas palabras en distinto orden. Predice si unigramas los distingue y compruébalo. Después prueba bigramas. Explica qué recuperan y qué no.
3. Calcula a mano el IDF de otro término del corpus diminuto con `log((1+n)/(1+df))+1`. Comprueba el valor con la biblioteca y explica por qué DF no cuenta repeticiones dentro de un documento.
4. Añade una palabra inventada solo a una copia de validación y demuestra que transformar no modifica vocabulario ni IDF. Muestra también un vector cero y su límite.

Entrega formativa: conserva notebook y tus explicaciones; no genera una tarea adicional. Orientación: 100–150 palabras más tablas/código. El tiempo restante de la práctica se reparte en carga (5 min), cuentas guiadas (15 min) y corpus/OOV/exportación (10 min).

In [6]:
indice=5
fila=X_val.getrow(indice)
print(val.iloc[indice][["id","text_processed"]].to_dict())
print(pd.DataFrame({"termino":vectorizador.get_feature_names_out()[fila.indices],"peso":fila.data}).sort_values("peso",ascending=False).head(3))
par=["pago cuenta bloqueada","cuenta pago bloqueada"]
u=CountVectorizer();g=CountVectorizer(ngram_range=(1,2))
U=u.fit_transform(par).toarray();G=g.fit_transform(par).toarray()
assert np.array_equal(U[0],U[1]) and not np.array_equal(G[0],G[1])
print("Iguales con unigramas:",np.array_equal(U[0],U[1]))
print("Iguales con bigramas:",np.array_equal(G[0],G[1]))
manual=np.log(4/2)+1
assert np.isclose(manual,tf.idf_[list(tf.get_feature_names_out()).index("tarjeta")])
antes=vectorizador.idf_.copy();vocab=vectorizador.vocabulary_.copy()
vectorizador.transform(["hiperfoliozeta "+val.iloc[0].text_processed])
assert np.array_equal(antes,vectorizador.idf_) and vocab==vectorizador.vocabulary_
completados.add(2)

{'id': 'S030', 'text_processed': 'El autenticador muestra un código inválido; ya reinicié y sigue igual, urge tantito.'}
   termino      peso
0   código  0.484429
3  muestra  0.484429
7       un  0.378452
Iguales con unigramas: True
Iguales con bigramas: False


# Ejemplo de resolución del reto
Se comparan dos secuencias con idénticos conteos: unigramas no registra orden, mientras que el bigrama cuenta quién aparece junto a quién. Esto no constituye comprensión sintáctica completa. El DF de tarjeta es 1 aunque una palabra pudiera repetirse dentro de su documento; su IDF es log(4/2)+1. Una palabra no observada no genera una nueva columna al transformar. Un vector cero indica falta de características conocidas, no ausencia de intención.

## 4. Guardar evidencia
Guardamos la tabla de una fila y versiones; no se guarda un clasificador en esta clase. La exportación del Pipeline completo se enseña en Clase 4.

In [7]:
tabla_fila.to_csv(OUT/"fila_tfidf_guiada.csv",index=False)
(OUT/"versiones.json").write_text(json.dumps(versiones,indent=2))
print("Reto completado:",2 in completados)

Reto completado: True


## Cierre
Explica una columna, una fila, una palabra OOV y por qué no se ajusta sobre test. En Clase 4 un clasificador aprenderá a relacionar estos números con etiquetas.

Referencias: [extracción de texto](https://scikit-learn.org/1.6/modules/feature_extraction.html), [TfidfVectorizer](https://scikit-learn.org/1.6/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).